# Does the Grassmann geometry matter?

An ablation of *Attention Is Not What You Need: Grassmann Flows* (arXiv 2512.19428) on Wikitext-2, for a Colab **L4** GPU.

Eight ~12.6M-parameter language models, identical except for the token mixer:

| mixer | what it mixes |
|---|---|
| `grassmann` | the paper's layer: Plücker coordinates (antisymmetric product) of pairs (t, t−Δ), Δ ∈ {1,2,4,8,12,16} |
| `sym` | same layer, symmetric product (no orientation) |
| `outer` | same layer, full product |
| `concat` | same layer, no product: just the two vectors side by side |
| `concat32` | exploratory: `concat` with the paper's r = 32 (64 features), added after the pilot |
| `hyena` | Hyena operator with a 16-step explicit filter |
| `swa` | sliding-window attention, window 16 |
| `attention` | full causal attention (the transformer baseline) |

The paper-style variants and the windowed baselines all see exactly 16 tokens back (checked by the tests), and every model except `concat32` (smaller by design) is within 3% of the attention baseline's parameter count.

**Plan:** tests → data → smoke run (measures speed on this GPU) → learning-rate sweep (1 seed) → main runs (3 seeds at each mixer's best LR) → table with 95% confidence intervals and paired same-seed comparisons.

The evaluation follows `PROTOCOL.md` in the repo, fixed before any run: choices on validation only, a declared confirmatory family with Holm correction, and a paired cluster bootstrap over test articles.

Results are saved under `/content/results` on this Colab machine. Every run cell skips runs that already finished, so if a cell is interrupted just run it again. `/content` is erased when the runtime is deleted, so download the results with the last cell before closing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Paths

In [ ]:
import os
REPO   = '/content/grassman'
DATA   = '/content/data'
OUTDIR = '/content/results'
RUNS   = f'{OUTDIR}/runs.jsonl'       # one JSON line per finished run
os.makedirs(OUTDIR, exist_ok=True)
print('results ->', RUNS)

## 2. Get the code from GitHub

In [ ]:
REPO_URL = 'https://github.com/thebnbrkr/grassman.git'
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q datasets transformers

## 3. Tests

The unit tests run on CPU in about a minute. They check the Plücker maths (definition, 3-D cross product, sign flip, plane invariance), that **no mixer reads future tokens** (the paper's equations as written pair t with t+Δ, which would leak), that each windowed mixer reaches exactly 16 tokens, parameter matching, determinism, the bootstrap and Holm maths, that the LR sweep never touches the test set, and the training/analysis code end to end. Do not continue if anything fails.

In [ ]:
!python -m pytest -q

## 4. Data: Wikitext-2-raw, bert-base-uncased WordPiece (as in the paper)

In [ ]:
!python -m grassman.data --data-dir {DATA}

## 5. Smoke run: speed on this GPU

200 steps per mixer. Prints throughput and the **estimated time of one full 30-epoch run** (≈ 69M training tokens), so you can budget the sweep and main runs before starting them.

In [ ]:
import json, subprocess, sys

def train(mixer, seed=0, lr=1e-3, epochs=30, tag='main', out=RUNS, extra=()):
    cmd = [sys.executable, '-m', 'grassman.train', '--mixer', mixer, '--seed', str(seed),
           '--lr', str(lr), '--epochs', str(epochs), '--tag', tag, '--data-dir', DATA,
           '--out', out, '--skip-existing', *extra]
    print('>>', ' '.join(cmd[3:]), flush=True)
    subprocess.run(cmd, cwd=REPO, check=True)

MIXERS = ['grassmann', 'sym', 'outer', 'concat', 'concat32', 'hyena', 'swa', 'attention']
SMOKE = '/content/smoke.jsonl'
if os.path.exists(SMOKE):
    os.remove(SMOKE)
for m in MIXERS:
    train(m, epochs=1, tag='smoke', out=SMOKE, extra=['--max-steps', '200', '--no-test'])

full_tokens = 30 * 2_303_695
print(f"\n{'mixer':<10} {'params':>8} {'tok/s':>10} {'est. 30-epoch run':>18}")
for line in open(SMOKE):
    r = json.loads(line)
    print(f"{r['config']['mixer']:<10} {r['params']/1e6:7.2f}M {r['tokens_per_s']:10,.0f} "
          f"{full_tokens / r['tokens_per_s'] / 60:15.1f} min")

## 6. Stage 1: learning-rate sweep

The paper does not report its optimizer settings, so each mixer gets its own best learning rate (one seed, 10 epochs), chosen on **validation** perplexity only; these runs never evaluate the test set (`--no-test`). This prevents any model from losing just because of a bad LR.

In [ ]:
LRS = [3e-4, 1e-3, 3e-3]
SWEEP_EPOCHS = 10
for m in MIXERS:
    for lr in LRS:
        train(m, seed=0, lr=lr, epochs=SWEEP_EPOCHS, tag='sweep', extra=['--no-test'])

In [ ]:
best = json.loads(subprocess.check_output(
    [sys.executable, '-m', 'grassman.analyze', RUNS, '--tag', 'sweep', '--best-lr'], cwd=REPO, text=True))
print(best)
!python -m grassman.analyze {RUNS} --tag sweep

## 7. Stage 2: main runs

3 seeds × 8 mixers, 30 epochs (as in the paper), each at its best LR. Test perplexity is taken at the epoch with the best validation perplexity.

The number of seeds is fixed by `PROTOCOL.md`. Do not add seeds after looking at the results: that biases the test.

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 30
for seed in SEEDS:                 # seed-major: a disconnect still leaves complete seeds
    for m in MIXERS:
        train(m, seed=seed, lr=best[m], epochs=EPOCHS, tag='main')

Part of the protocol (exploratory): the paper L2-normalises the Plücker vector. This checks whether that matters, at `grassmann`'s learning rate. It shows up as `grassmann-raw`.

In [ ]:
for seed in SEEDS:
    train('grassmann', seed=seed, lr=best['grassmann'], epochs=EPOCHS, tag='main', extra=['--no-normalize'])

## 8. Results

In [ ]:
!python -m grassman.analyze {RUNS} --tag main

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

curves = defaultdict(list)
for line in open(RUNS):
    r = json.loads(line)
    if r['tag'] != 'main':
        continue
    name = r['config']['mixer'] + ('' if r['config']['normalize'] else '-raw')
    curves[name].append([h['val_ppl'] for h in r['history']])

fig, ax = plt.subplots(figsize=(8, 5))
for name, runs in sorted(curves.items()):
    n = min(len(c) for c in runs)
    mean = [sum(c[i] for c in runs) / len(runs) for i in range(n)]
    ax.plot(range(1, n + 1), mean, label=f'{name} ({len(runs)} seeds)')
ax.set_xlabel('epoch'); ax.set_ylabel('validation perplexity (mean over seeds)')
ax.set_ylim(top=600); ax.legend(); ax.grid(alpha=0.3)
plt.savefig(f'{OUTDIR}/val_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read the results (decision rules from `PROTOCOL.md`)

- **Main table:** mean ± 95% t-interval over seeds (training noise).
- **Confirmatory family** (`sym`, `outer`, `concat`, `attention` vs `grassmann`): perplexity difference with a 95% cluster-bootstrap interval over test articles, the p-value, and the Holm-adjusted p-value.
- **"The geometry matters"** only if `sym`, `outer` and `concat` are all worse with Holm p < 0.05 **and** `grassmann` wins in every seed (the `wins` column shows 3/3). Otherwise report it as not shown, with the size of the difference.
- **Reproduction check:** the `vs attention` column against the paper (10–15% worse) and the reproduction (22.6%).
- **Exploratory rows** (`hyena`, `swa`, `grassmann-raw`) and the position and frequency breakdowns are reported but not used for claims.
- Record the outcome, including results that go against the paper, in `RESEARCH_LOG.md`.

Optional: download everything in `/content/results` before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')